# Phase 3: Workforce Intelligence — Step 3.7: Employee Intelligence Master Table

This notebook compiles our finalized master analytical table (`data/processed/employee_intelligence.csv`) by joining multiple data streams:
1. **Employee Demographics**: `employees.csv`
2. **Attrition Model Predictions**: `attrition_probability` and `risk_bucket` calculated using `models/v1/attrition_pipeline.joblib`
3. **Workforce Engagement Metrics**: `engagement_data.csv` (left-joined on `Employee ID` == `EmployeeNumber` where matchable, else kept as `null` per Key Decisions Log to avoid faking joins)
4. **Upskilling & Skill Gap Metrics**: `total_skill_gap_count`, `skill_gap_list`, and the `top_course_recommendation` (the first ranked course from `employee_recommendations_pivoted.csv` generated in Step 3.6).

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

proc_dir = os.path.join("data", "processed")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

## 1. Load Processed Datasets

In [2]:
df_emp = pd.read_csv(os.path.join(proc_dir, "employees.csv"))
df_eng = pd.read_csv(os.path.join(proc_dir, "engagement_data.csv"))
df_gaps = pd.read_csv(os.path.join(proc_dir, "employee_skill_gaps.csv"))
df_recs = pd.read_csv(os.path.join(proc_dir, "employee_recommendations_pivoted.csv"))

print(f"Employees Shape: {df_emp.shape}")
print(f"Engagement Shape: {df_eng.shape}")
print(f"Skill Gaps Shape: {df_gaps.shape}")
print(f"Recommendations Shape: {df_recs.shape}")

Employees Shape: (1470, 35)
Engagement Shape: (2843, 5)
Skill Gaps Shape: (48468, 5)
Recommendations Shape: (1470, 4)


## 2. Load Attrition Model & Compute Predictions
We load the versioned balanced Logistic Regression pipeline and compute attrition probabilities on the raw unscaled encoded features.

In [ ]:
from app.ml.predictor import get_risk_bucket, predict_attrition_probability

# Use the same validated production path as the direct predictor check instead of
# reimplementing feature engineering in the notebook.
df_attrition = pd.DataFrame({
    "employee_id": df_emp["EmployeeNumber"],
    "attrition_probability": [
        predict_attrition_probability(row)
        for _, row in df_emp.iterrows()
    ]
})
df_attrition["risk_bucket"] = df_attrition["attrition_probability"].apply(get_risk_bucket)

print("Risk bucket value counts:")
print(df_attrition["risk_bucket"].value_counts())
print("Probability summary:")
print(df_attrition["attrition_probability"].describe())

Risk bucket value counts:
risk_bucket
Low       895
Medium    329
High      246
Name: count, dtype: int64


## 3. Aggregate and Merge Engagement Metrics
We group engagement records by `Employee ID` and left-join them (where matchable) to avoid faking joins for the unlinkable demographic portion.

In [4]:
df_eng_uniq = df_eng.groupby("Employee ID").agg(
    avg_engagement_score=("Engagement Score", "mean"),
    avg_satisfaction_score=("Satisfaction Score", "mean"),
    avg_work_life_balance_score=("Work-Life Balance Score", "mean")
).reset_index()

# Initialize master table
df_master = df_emp[["EmployeeNumber", "JobRole", "Department"]].rename(columns={"EmployeeNumber": "employee_id"})
df_master = df_master.merge(df_attrition, on="employee_id", how="left")
df_master = df_master.merge(df_eng_uniq, left_on="employee_id", right_on="Employee ID", how="left").drop(columns=["Employee ID"])

print(f"Master after demographics and engagement joins: {df_master.shape}")

Master after demographics and engagement joins: (1470, 8)


## 4. Merge Skill Gaps and Recommendations

In [5]:
# Group skill gaps
df_gaps_agg = df_gaps.groupby("employee_id").agg(
    total_skill_gap_count=("missing_skill", "count"),
    skill_gap_list=("missing_skill", lambda x: ", ".join(list(x)))
).reset_index()

df_master = df_master.merge(df_gaps_agg, on="employee_id", how="left")
df_master["total_skill_gap_count"] = df_master["total_skill_gap_count"].fillna(0).astype(int)
df_master["skill_gap_list"] = df_master["skill_gap_list"].fillna("")

# Merge course recommendations
df_master = df_master.merge(df_recs[["employee_id", "recommended_course_1"]], on="employee_id", how="left")
df_master = df_master.rename(columns={"recommended_course_1": "top_course_recommendation"})

print(f"Final master table shape: {df_master.shape}")
print(df_master.head(5))

Final master table shape: (1470, 11)
   employee_id  ...                         top_course_recommendation
0            1  ...  Critical Thinking and Analytical Problem Solving
1            2  ...  Critical Thinking and Analytical Problem Solving
2            4  ...                   Speed Reading and Comprehension
3            5  ...                FileMaker Pro Database Development
4            7  ...                FileMaker Pro Database Development

[5 rows x 11 columns]


## 5. Export Master Intelligence Table
We export this table to `data/processed/employee_intelligence.csv` which is read directly by the downstream reporting applications and dashboards.

In [6]:
output_path = os.path.join(proc_dir, "employee_intelligence.csv")
df_master.to_csv(output_path, index=False)
print(f"Successfully saved Employee Intelligence Table to {output_path}")

Successfully saved Employee Intelligence Table to data\processed\employee_intelligence.csv
